# Train a Cat vs Dog Classifier with Transfer Learning

This notebook uses EfficientNetB0 pretrained on ImageNet to build a high-accuracy Cat vs Dog classifier. The workflow starts with a frozen backbone, then performs a short fine-tuning stage by unfreezing the last 25 layers.

In [7]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.data_utils import build_datasets, set_seed
from src.model_utils import (
    build_callbacks,
    create_base_model,
    evaluate_model,
    make_confusion_matrix,
    plot_training_history,
    save_model_artifacts,
)

set_seed(42)

project_root = Path.cwd().resolve()
dataset_root = project_root / "PetImages"
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")
print(f"Project root: {project_root}")
print(f"Dataset root: {dataset_root}")

TensorFlow version: 2.21.0
Project root: C:\Users\taikh\OneDrive\Documents\Dogs and Cats Classifier\cat_dog_classifier\notebooks
Dataset root: C:\Users\taikh\OneDrive\Documents\Dogs and Cats Classifier\cat_dog_classifier\notebooks\PetImages


## Load Prepared Datasets

The dataset was prepared in the previous notebook. This cell loads the training, validation, and test sets using the reusable utilities.

In [ ]:
train_ds, val_ds, test_ds, class_names = build_datasets(
    dataset_dir=dataset_root,
    image_size=(224, 224),
    batch_size=32,
    seed=42,
)

print(f"Training batches: {len(train_ds)}")
print(f"Validation batches: {len(val_ds)}")
print(f"Test batches: {len(test_ds)}")
print(f"Class names: {class_names}")

## Build the Initial Transfer Learning Model

The pretrained EfficientNetB0 backbone is frozen at first so the classifier head learns quickly without changing the feature extractor.

In [ ]:
model = create_base_model(input_shape=(224, 224, 3))
model.summary()

## Train the Top Layer First

The initial training phase optimizes the new classifier head using a relatively high learning rate and early stopping.

In [ ]:
checkpoint_path = models_dir / "best_initial_model.keras"
callbacks = build_callbacks(checkpoint_path)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1,
)

## Fine-Tune the Top EfficientNet Layers

After the initial training, the last 25 layers of EfficientNetB0 are unfrozen for a second training pass with a much smaller learning rate.

In [ ]:
base_model = model.get_layer("efficientnetb0")
for layer in base_model.layers[:-25]:
    layer.trainable = False
for layer in base_model.layers[-25:]:
    layer.trainable = True

for layer in base_model.layers:
    if hasattr(layer, "batch_normalization"):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=build_callbacks(models_dir / "best_fine_tuned_model.keras"),
    verbose=1,
)

history.history.update({
    key: history.history.get(key, []) + fine_tune_history.history.get(key, [])
    for key in set(history.history) | set(fine_tune_history.history)
})

## Plot Training Curves

Training curves help confirm that the model improved over time and that the learning-rate schedule was effective.

In [ ]:
plot_training_history(history, models_dir / "training_history.png")
plt.show()

## Evaluate the Final Model

The test set is used to compute the final metrics and generate a confusion matrix for deeper inspection.

In [ ]:
evaluation = evaluate_model(model, test_ds)
print("Test accuracy:", evaluation["test_accuracy"])
print("Precision:", evaluation["precision"])
print("Recall:", evaluation["recall"])
print("F1 score:", evaluation["f1_score"])

confusion = make_confusion_matrix(model, test_ds, class_names)
print("Confusion matrix:")
print(confusion)

## Save the Final Model

The trained model and class names are saved so they can be reused later for inference or deployment.

In [ ]:
save_model_artifacts(model, models_dir, np.array(class_names))
print(f"Saved model artifacts to {models_dir}")